In [ ]:
import sys

# repo root, relative to this notebook (utils/diversity_analysis/)
sys.path.append("../..")

from models.api_model import OpenRouterModel
from utils.json_utils import extract_json
from pathlib import Path
from collections import Counter
from tqdm import tqdm

PROMPT_PATH = "prompt.txt" 



In [70]:
def read_cipher_infos(root_dir: str) -> list[dict]:
    root = Path(root_dir)
    results = []

    for trial_dir in root.rglob("attack_trial_*"):
        if not trial_dir.is_dir():
            continue

        cipher_path = trial_dir / "cipher_info.txt"

        if not cipher_path.exists():
            print(f"Warning: no cipher_info.txt in {trial_dir}")
            continue

        with open(cipher_path, "r", encoding="utf-8") as f:
            cipher_info = f.read()

        results.append({
            "trial": trial_dir.name,
            "path": str(cipher_path),
            "cipher_info": cipher_info,
        })

    return results

In [71]:
class Analyser:
    def __init__(self):
        self.judge = OpenRouterModel(model_name="openai/gpt-4o-mini", reasoning=False)
        with open(PROMPT_PATH, "r") as f:
            prompt = f.read()
        self.prompt = prompt
    

    def prepare_stats(self, root_dir):
        cipher_infos = read_cipher_infos(root_dir)
        category_counts = Counter()
        mechanism_count = Counter()
        composite_counts = Counter()

        results_categories = []

        for cipher_info in tqdm(cipher_infos):
            cipher_info = cipher_info["cipher_info"]

            raw_json = self.judge.ask(cipher_info, self.prompt)
            json_dict = extract_json(raw_json)

            categories = json_dict.get("categories", [])
            is_composite = json_dict.get("is_composite", False)

            results_categories.append(categories)

            # Count each category once for this cipher
            for category in set(categories):
                category_counts[category] += 1

            # Number of mechanisms used by this cipher
            mechanism_count[len(set(categories))] += 1

            # Composite / non-composite
            composite_counts[is_composite] += 1

        return {
            "n_ciphers": len(cipher_infos),
            "category_counts": category_counts,
            "mechanism_count": mechanism_count,
            "composite_counts": composite_counts,
            "all_categories": results_categories,
        }

In [115]:
import matplotlib.pyplot as plt
import numpy as np


ALL_CATEGORIES = [
    "monoalphabetic_substitution",
    "polyalphabetic_substitution",
    "homophonic_substitution",
    "homoglyph_substitution",
    "encoding_representation",
    "transposition_permutation",
    "reversal",
    "insertion_noise",
    "deletion_lossy",
    "numeric_transformation",
    "word_token_level_transformation",
    "other",
]


def plot_cipher_category_matrix(
    all_categories,
    save_path="cipher_category_matrix.pdf",
    seed=42,
):
    """
    Plot a matrix showing the diversity of cipher mechanisms.

    Diagonal cells:
        Ciphers using a single mechanism.

    Upper-triangular cells:
        Ciphers combining two mechanisms.

    Lower-triangular cells:
        Unused, because mechanism pairs are unordered.

    Ciphers with 3+ mechanisms are omitted.

    All predefined categories are shown, even if they do not occur
    in the current experiment.
    """

    # ---------------------------------------------------------
    # Fixed category order for both axes
    # ---------------------------------------------------------
    categories = ALL_CATEGORIES.copy()

    category_to_idx = {
        category: i
        for i, category in enumerate(categories)
    }

    # ---------------------------------------------------------
    # Count ciphers in each cell
    # ---------------------------------------------------------
    cells = {}

    for cipher_categories in all_categories:

        # Remove duplicate categories within one cipher
        cipher_categories = list(dict.fromkeys(cipher_categories))

        # Ignore unknown categories
        cipher_categories = [
            c for c in cipher_categories
            if c in category_to_idx
        ]

        # Single-mechanism cipher -> diagonal
        if len(cipher_categories) == 1:
            idx = category_to_idx[cipher_categories[0]]
            cell = (idx, idx)

        # Two-mechanism cipher -> upper triangle
        elif len(cipher_categories) == 2:
            c1, c2 = sorted(
                cipher_categories,
                key=lambda c: category_to_idx[c]
            )

            cell = (
                category_to_idx[c1],
                category_to_idx[c2]
            )

        # 3+ mechanisms are not shown
        else:
            continue

        cells[cell] = cells.get(cell, 0) + 1

    rng = np.random.default_rng(seed)

    fig, ax = plt.subplots(figsize=(11, 10))

    n = len(categories)

    # ---------------------------------------------------------
    # Shade unused lower-triangular cells
    # ---------------------------------------------------------
    for x in range(n):
        for y in range(n):
            if y < x:
                rect = plt.Rectangle(
                    (x - 0.5, y - 0.5),
                    1,
                    1,
                    facecolor="lightgray",
                    alpha=0.7,
                    edgecolor="none",
                    zorder=0,
                )
                ax.add_patch(rect)

    # ---------------------------------------------------------
    # Highlight diagonal cells
    # ---------------------------------------------------------
    for i in range(n):
        rect = plt.Rectangle(
            (i - 0.5, i - 0.5),
            1,
            1,
            facecolor="lightskyblue",
            alpha=0.30,
            edgecolor="none",
            zorder=1,
        )
        ax.add_patch(rect)

    # ---------------------------------------------------------
    # Draw cell grid
    # ---------------------------------------------------------
    ax.set_xticks(
        np.arange(-0.5, n, 1),
        minor=True,
    )

    ax.set_yticks(
        np.arange(-0.5, n, 1),
        minor=True,
    )

    ax.grid(
        which="minor",
        linewidth=0.7,
        alpha=0.35,
        color="gray",
        zorder=2,
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False,
    )

    # ---------------------------------------------------------
    # Staircase boundary along CELL BORDERS
    #
    # This separates:
    #   lower triangle = unused
    #   diagonal       = single mechanism
    #   upper triangle = two mechanisms
    # ---------------------------------------------------------
    staircase_x = [0.5]
    staircase_y = [-0.5]

    for i in range(n - 1):
        # Move up
        staircase_x.append(i + 0.5)
        staircase_y.append(i + 0.5)

        # Move right
        staircase_x.append(i + 1.5)
        staircase_y.append(i + 0.5)

    ax.plot(
        staircase_x,
        staircase_y,
        color="black",
        linewidth=2,
        alpha=0.65,
        zorder=5,
    )

    # ---------------------------------------------------------
    # Plot individual ciphers
    # ---------------------------------------------------------
    for (x, y), count in cells.items():

        # Spread multiple ciphers inside the same cell
        xs = x + rng.uniform(-0.27, 0.27, count)
        ys = y + rng.uniform(-0.27, 0.27, count)

        ax.scatter(
            xs,
            ys,
            s=45,
            alpha=0.75,
            zorder=6,
        )

    # ---------------------------------------------------------
    # Pretty category labels
    # ---------------------------------------------------------
    pretty_labels = [
        category.replace("_", " ").title()
        for category in categories
    ]

    # Same order on both axes
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))

    ax.set_xticklabels(
        pretty_labels,
        rotation=45,
        ha="right",
        fontsize=13,
    )

    ax.set_yticklabels(
        pretty_labels,
        fontsize=13,
    )

    # ---------------------------------------------------------
    # Axis limits
    # ---------------------------------------------------------
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(-0.5, n - 0.5)

    # Standard Cartesian orientation:
    # (0, 0) is bottom-left.
    # Diagonal goes bottom-left -> top-right.

    ax.tick_params(
        axis="both",
        labelsize=13,
    )

    plt.tight_layout()

    # ---------------------------------------------------------
    # Save as vector PDF
    # ---------------------------------------------------------
    fig.savefig(
        save_path,
        format="pdf",
        bbox_inches="tight",
    )

    return fig, ax

In [ ]:
analyser = Analyser()

results = analyser.prepare_stats("../../logs_final/dynamic/harmbench_split/adaptive_with_guards/gemini_3_5_flash_")


plot_cipher_category_matrix(results["all_categories"], save_path="results/guards/gemini.pdf")
print(results)

In [ ]:
analyser = Analyser()

results = analyser.prepare_stats("../../logs_final/dynamic/harmbench_split/adaptive_with_guards/kimi_k3_")


plot_cipher_category_matrix(results["all_categories"], save_path="results/guards/kimi.pdf")
print(results)

In [ ]:
analyser = Analyser()

results = analyser.prepare_stats("../../logs_final/dynamic/harmbench_split/adaptive_with_guards/llama_4_maverick_")


plot_cipher_category_matrix(results["all_categories"], save_path="results/guards/llama.pdf")
print(results)

In [ ]:
analyser = Analyser()

results = analyser.prepare_stats("../../logs_final/dynamic/harmbench_split/adaptive_with_guards/qwen3_5_397b_a17b_")


plot_cipher_category_matrix(results["all_categories"], save_path="results/guards/qwen.pdf")
print(results)

## No guards

In [ ]:
analyser = Analyser()

results = analyser.prepare_stats("../../logs_final/dynamic/harmbench_split/adaptive_no_guards/gemini_3_5_flash_")


plot_cipher_category_matrix(results["all_categories"], save_path="results/no_guards/gemini.pdf")
print(results)

In [ ]:
analyser = Analyser()

results = analyser.prepare_stats("../../logs_final/dynamic/harmbench_split/adaptive_no_guards/kimi_k3_")


plot_cipher_category_matrix(results["all_categories"], save_path="results/no_guards/kimi.pdf")
print(results)

In [ ]:
analyser = Analyser()

results = analyser.prepare_stats("../../logs_final/dynamic/harmbench_split/adaptive_no_guards/llama_4_maverick_")


plot_cipher_category_matrix(results["all_categories"], save_path="results/no_guards/llama.pdf")
print(results)

In [ ]:
analyser = Analyser()

results = analyser.prepare_stats("../../logs_final/dynamic/harmbench_split/adaptive_no_guards/qwen3_5_397b_a17b_")


plot_cipher_category_matrix(results["all_categories"], save_path="results/no_guards/qwen.pdf")
print(results)